# Assay Ground Truth Analysis and TADP-Based Stratified Sampling

This notebook analyses ground truth JSON files for assay extraction and creates
stratified splits based on **Total Assay Data Points (TADP)**.

**TADP** counts individual extractable assay values from:
- `isolates_with_linking` (per-isolate assay data)
- `no_isolates_only_assayinformation` (study-level assay data)

**Note:** `isolate_without_linking` contains only isolate IDs without assay data, so it is NOT counted in TADP.

---

**Author:** Luqman  
**Project:** AI6129 Pathogen Tracking System  
**Date:** January 2025

In [1]:
import re
import json
import random
from pathlib import Path
from collections import Counter
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any
from google.colab import drive

In [3]:
# Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive', force_remount=False)
print("Google Drive mounted")

Mounting Google Drive...
Mounted at /content/drive
Google Drive mounted


In [4]:
# =============================================================================
# CONFIGURATION - UPDATE THESE PATHS FOR YOUR ENVIRONMENT
# =============================================================================

# Directory paths
GROUND_TRUTH_FOLDER = "/content/drive/MyDrive/AI6129/ground_truth"
OUTPUT_FOLDER = "/content/drive/MyDrive/AI6129/assay/validation_splits"

# Assay fields to count for TADP calculation
ASSAY_FIELDS = [
    "serotype",
    "mlst",
    "spi",
    "amr",
    "plasmid",
    "snp",
    "virulence_genes"
]

# TADP-based stratification bins
STRATA_BINS = {
    "zero": (0, 0),             # 0 TADP (negative examples - no assay data)
    "low": (1, 10),             # 1-10 TADP (simple documents)
    "medium": (11, 50),         # 11-50 TADP (moderate complexity)
    "high": (51, float('inf'))  # 51+ TADP (complex documents)
}

# Split ratios (must sum to 1.0)
SPLIT_RATIOS = {
    "development": 0.20,    # ~45 documents (rapid iteration)
    "validation": 0.51,     # ~116 documents (model comparison)
    "test": 0.29            # ~66 documents (held-out final evaluation)
}

# Documents to pin to specific splits (e.g., extreme outliers)
PINNED_DOCUMENTS = {
    "development": [],
    "validation": [],
    "test": []
}

RANDOM_SEED = 42  # For reproducibility

## TADP Calculation Functions

These functions calculate the Total Assay Data Points (TADP) for each document.

In [5]:
def count_ast_entries(ast_data: List[dict]) -> int:
    """
    Count the number of antibiotic entries in AST data.

    Args:
        ast_data: List of AST dictionaries, each containing 'Antibiotics' list

    Returns:
        Total count of antibiotic entries
    """
    total = 0

    if not ast_data or not isinstance(ast_data, list):
        return 0

    for ast_entry in ast_data:
        if isinstance(ast_entry, dict):
            antibiotics = ast_entry.get("Antibiotics", [])
            if isinstance(antibiotics, list):
                total += len(antibiotics)

    return total


def count_assay_fields_in_isolate(isolate: dict) -> int:
    """
    Count assay data points in a single isolate entry.

    Args:
        isolate: Dictionary containing isolate data

    Returns:
        Count of assay data points
    """
    count = 0

    if not isinstance(isolate, dict):
        return 0

    # Count standard assay fields (list-based)
    for field in ASSAY_FIELDS:
        val = isolate.get(field, [])
        if isinstance(val, list):
            count += len(val)
        elif isinstance(val, str) and val and val.lower() != "null":
            count += 1

    # Count AST entries (nested structure)
    count += count_ast_entries(isolate.get("ast_data", []))

    return count


def calculate_tadp(ground_truth_data: dict) -> int:
    """
    Calculate Total Assay Data Points (TADP) from ground truth.

    Counts individual values from:
    - serotype, mlst, spi, amr, plasmid, snp, virulence_genes (list lengths)
    - ast_data (count of antibiotic entries)

    Sources:
    - isolates_with_linking (per-isolate assay data)
    - no_isolates_only_assayinformation (study-level assay data)

    Does NOT count isolate_without_linking (just IDs, no assay data).

    Args:
        ground_truth_data: The loaded JSON ground truth dictionary

    Returns:
        Total Assay Data Points count
    """
    total = 0

    # Count from isolates_with_linking
    isolates_with_linking = ground_truth_data.get("isolates_with_linking", [])
    if isinstance(isolates_with_linking, list):
        for isolate in isolates_with_linking:
            total += count_assay_fields_in_isolate(isolate)

    # Count from no_isolates_only_assayinformation
    nioai = ground_truth_data.get("no_isolates_only_assayinformation", {})
    if isinstance(nioai, dict) and nioai:
        total += count_assay_fields_in_isolate(nioai)

    return total


def categorise_document_sections(ground_truth_data: dict) -> str:
    """
    Categorise document by which sections contain data.

    Categories:
    - IWL: Only isolates_with_linking has data
    - IWOL: Only isolate_without_linking has data
    - NIOAI: Only no_isolates_only_assayinformation has data
    - IWL+NIOAI: Both linked and study-level assay data
    - MIXED: Multiple sections with data
    - EMPTY: No sections have data

    Args:
        ground_truth_data: The loaded JSON ground truth dictionary

    Returns:
        Category string
    """
    has_iwl = False
    has_iwol = False
    has_nioai = False

    # Check isolates_with_linking
    iwl = ground_truth_data.get("isolates_with_linking", [])
    if isinstance(iwl, list) and len(iwl) > 0:
        has_iwl = True

    # Check isolate_without_linking
    iwol = ground_truth_data.get("isolate_without_linking", [])
    if isinstance(iwol, list) and len(iwol) > 0:
        has_iwol = True

    # Check no_isolates_only_assayinformation
    nioai = ground_truth_data.get("no_isolates_only_assayinformation", {})
    if isinstance(nioai, dict) and bool(nioai):
        has_nioai = True

    # Determine category
    categories = []
    if has_iwl:
        categories.append("IWL")
    if has_iwol:
        categories.append("IWOL")
    if has_nioai:
        categories.append("NIOAI")

    if not categories:
        return "EMPTY"
    elif len(categories) == 1:
        return categories[0]
    else:
        return "+".join(categories)


def get_tadp_stratum(tadp: int) -> str:
    """
    Determine which stratum a document belongs to based on TADP.

    Args:
        tadp: Total Assay Data Points count

    Returns:
        Stratum name: "zero", "low", "medium", or "high"
    """
    for stratum_name, (min_val, max_val) in STRATA_BINS.items():
        if min_val <= tadp <= max_val:
            return stratum_name

    return "high"  # Default fallback

## Data Loading Functions

In [6]:
def extract_pmcid_from_filename(filename: str) -> Optional[str]:
    """
    Extract PMC ID from JSON filename.

    Expected patterns:
    - PMC1234567.json
    - PMC1234567_metadata.json
    - pmc1234567.json (case-insensitive)

    Args:
        filename: The JSON filename

    Returns:
        PMC ID string (e.g., "PMC1234567") or None if not found
    """
    pattern = r'(PMC\d+)'
    match = re.search(pattern, filename, re.IGNORECASE)

    if match:
        return match.group(1).upper()

    return None


def load_all_ground_truth(folder_path: str) -> Dict[str, dict]:
    """
    Load all JSON ground truth files and calculate TADP for each.

    Args:
        folder_path: Path to the ground truth folder

    Returns:
        Dictionary mapping PMCID to document data with TADP
    """
    ground_truth_dict = {}
    errors = []

    json_files = list(Path(folder_path).glob("*.json"))
    print(f"Found {len(json_files)} JSON files in folder")

    for gt_file in json_files:
        try:
            with open(gt_file, 'r', encoding='utf-8') as f:
                data = json.load(f)

            # Extract PMCID
            pmcid = data.get("pmcid") or extract_pmcid_from_filename(gt_file.name)

            if not pmcid:
                errors.append(f"Could not extract PMCID from: {gt_file.name}")
                continue

            # Calculate TADP
            tadp = calculate_tadp(data)
            stratum = get_tadp_stratum(tadp)
            category = categorise_document_sections(data)

            # Count isolates and other metrics
            iwl_count = len(data.get("isolates_with_linking", []))
            iwol_count = len(data.get("isolate_without_linking", []))
            clean_text_length = len(data.get("clean_text", ""))

            ground_truth_dict[pmcid] = {
                "pmcid": pmcid,
                "filename": gt_file.name,
                "filepath": str(gt_file),
                "tadp": tadp,
                "stratum": stratum,
                "category": category,
                "isolates_with_linking_count": iwl_count,
                "isolate_without_linking_count": iwol_count,
                "clean_text_length": clean_text_length,
                "raw_data": data
            }

        except json.JSONDecodeError as e:
            errors.append(f"JSON parse error in {gt_file.name}: {e}")
        except Exception as e:
            errors.append(f"Error loading {gt_file.name}: {e}")

    if errors:
        print(f"\nWarning: {len(errors)} files had errors:")
        for err in errors[:10]:
            print(f"  - {err}")
        if len(errors) > 10:
            print(f"  ... and {len(errors) - 10} more")

    return ground_truth_dict

## Analysis Functions

In [7]:
def analyse_tadp_distribution(ground_truth_dict: Dict[str, dict]) -> dict:
    """
    Analyse the distribution of TADP across all documents.

    Args:
        ground_truth_dict: Dictionary from load_all_ground_truth()

    Returns:
        Statistics dictionary with distribution data
    """
    statistics = {
        "total_documents": len(ground_truth_dict),
        "tadp_distribution": [],
        "strata_counts": Counter(),
        "strata_documents": {stratum: [] for stratum in STRATA_BINS.keys()},
        "category_counts": Counter(),
        "category_documents": {},
        "zero_tadp_count": 0,
        "max_tadp": 0,
        "min_tadp": float('inf'),
        "total_tadp": 0,
        "mean_tadp": 0,
    }

    for pmcid, doc_data in ground_truth_dict.items():
        tadp = doc_data["tadp"]
        stratum = doc_data["stratum"]
        category = doc_data["category"]

        # TADP statistics
        statistics["tadp_distribution"].append(tadp)
        statistics["strata_counts"][stratum] += 1
        statistics["strata_documents"][stratum].append(pmcid)
        statistics["total_tadp"] += tadp

        # Category statistics
        statistics["category_counts"][category] += 1
        if category not in statistics["category_documents"]:
            statistics["category_documents"][category] = []
        statistics["category_documents"][category].append(pmcid)

        if tadp == 0:
            statistics["zero_tadp_count"] += 1

        if tadp > statistics["max_tadp"]:
            statistics["max_tadp"] = tadp

        if tadp < statistics["min_tadp"]:
            statistics["min_tadp"] = tadp

    # Calculate mean
    if statistics["total_documents"] > 0:
        statistics["mean_tadp"] = (
            statistics["total_tadp"] / statistics["total_documents"]
        )

    # Calculate count frequency
    count_freq = Counter(statistics["tadp_distribution"])
    statistics["tadp_frequency"] = dict(sorted(count_freq.items()))

    return statistics


def print_distribution_report(statistics: dict) -> None:
    """
    Print a formatted report of the TADP distribution.

    Args:
        statistics: Statistics dictionary from analyse_tadp_distribution()
    """
    print("\n" + "=" * 70)
    print("ASSAY GROUND TRUTH TADP DISTRIBUTION ANALYSIS")
    print("=" * 70)

    print(f"\nTotal Documents: {statistics['total_documents']}")
    print(f"Total Assay Data Points: {statistics['total_tadp']}")
    print(f"Mean TADP per Document: {statistics['mean_tadp']:.2f}")
    print(f"Min TADP: {statistics['min_tadp']}")
    print(f"Max TADP: {statistics['max_tadp']}")
    print(f"Documents with Zero TADP: {statistics['zero_tadp_count']}")

    print("\n" + "-" * 70)
    print("TADP STRATIFICATION SUMMARY")
    print("-" * 70)
    print(f"{'Stratum':<12} {'TADP Range':<15} {'Count':<10} {'Percentage':<12}")
    print("-" * 70)

    total = statistics["total_documents"]

    for stratum_name, (min_val, max_val) in STRATA_BINS.items():
        count = statistics["strata_counts"].get(stratum_name, 0)
        pct = (count / total * 100) if total > 0 else 0
        if max_val == float('inf'):
            range_str = f"{min_val}+"
        elif min_val == max_val:
            range_str = f"{min_val}"
        else:
            range_str = f"{min_val}-{int(max_val)}"
        print(f"{stratum_name:<12} {range_str:<15} {count:<10} {pct:.1f}%")

    print("\n" + "-" * 70)
    print("DOCUMENT CATEGORY DISTRIBUTION")
    print("-" * 70)
    print(f"{'Category':<20} {'Count':<10} {'Percentage':<12} {'Description'}")
    print("-" * 70)

    category_descriptions = {
        "IWL": "Isolates with linked assay data",
        "IWOL": "Isolate IDs only (no assay data)",
        "NIOAI": "Study-level assay data (no isolate linking)",
        "IWL+IWOL": "Linked isolates + additional IDs",
        "IWL+NIOAI": "Both isolate-linked and study-level data",
        "EMPTY": "No data in any section"
    }

    for category, count in sorted(statistics["category_counts"].items(),
                                   key=lambda x: -x[1]):
        pct = (count / total * 100) if total > 0 else 0
        desc = category_descriptions.get(category, "Mixed sections")
        print(f"{category:<20} {count:<10} {pct:.1f}%        {desc}")

    print("\n" + "-" * 70)
    print("DETAILED TADP FREQUENCY (first 25 values)")
    print("-" * 70)
    print(f"{'TADP':<10} {'Documents':<15} {'Percentage':<12}")
    print("-" * 70)

    for tadp_val, freq in sorted(statistics["tadp_frequency"].items())[:25]:
        pct = (freq / total * 100) if total > 0 else 0
        print(f"{tadp_val:<10} {freq:<15} {pct:.1f}%")

    if len(statistics["tadp_frequency"]) > 25:
        print(f"... and {len(statistics['tadp_frequency']) - 25} more unique TADP values")

## Stratified Sampling Functions

In [8]:
def create_stratified_three_set_split(
    ground_truth_dict: Dict[str, dict],
    split_ratios: Dict[str, float] = None,
    random_seed: int = RANDOM_SEED,
    pinned_documents: Dict[str, List[str]] = None
) -> Dict[str, List[str]]:
    """
    Create stratified development/validation/test splits based on TADP.

    Maintains proportional representation across all TADP strata.

    Args:
        ground_truth_dict: Dictionary from load_all_ground_truth()
        split_ratios: Dict with 'development', 'validation', 'test' ratios
        random_seed: For reproducibility
        pinned_documents: Dict of PMCIDs to pin to specific splits

    Returns:
        Dictionary with lists of PMCIDs for each split
    """
    if split_ratios is None:
        split_ratios = SPLIT_RATIOS

    if pinned_documents is None:
        pinned_documents = PINNED_DOCUMENTS

    # Set random seed
    random.seed(random_seed)

    # Step 1: Separate pinned from unpinned documents
    pinned_dev = []
    pinned_val = []
    pinned_test = []
    unpinned_pmcids = []

    for pmcid in ground_truth_dict.keys():
        if pmcid in pinned_documents.get("development", []):
            pinned_dev.append(pmcid)
            print(f"Pinned to development: {pmcid}")
        elif pmcid in pinned_documents.get("validation", []):
            pinned_val.append(pmcid)
            print(f"Pinned to validation: {pmcid}")
        elif pmcid in pinned_documents.get("test", []):
            pinned_test.append(pmcid)
            print(f"Pinned to test: {pmcid}")
        else:
            unpinned_pmcids.append(pmcid)

    # Warn about pinned documents not found
    all_pinned = (
        pinned_documents.get("development", []) +
        pinned_documents.get("validation", []) +
        pinned_documents.get("test", [])
    )
    for pmcid in all_pinned:
        if pmcid not in ground_truth_dict:
            print(f"Warning: Pinned document {pmcid} not found in ground truth")

    # Step 2: Group unpinned documents by stratum
    strata_groups = {stratum: [] for stratum in STRATA_BINS.keys()}

    for pmcid in unpinned_pmcids:
        stratum = ground_truth_dict[pmcid]["stratum"]
        strata_groups[stratum].append(pmcid)

    # Shuffle within each stratum
    for stratum in strata_groups:
        random.shuffle(strata_groups[stratum])

    # Step 3: Allocate from each stratum to splits
    splits = {
        "development": list(pinned_dev),
        "validation": list(pinned_val),
        "test": list(pinned_test)
    }

    for stratum, pmcids in strata_groups.items():
        n_total = len(pmcids)

        if n_total == 0:
            continue

        # Calculate split sizes for this stratum
        n_dev = int(round(n_total * split_ratios["development"]))
        n_test = int(round(n_total * split_ratios["test"]))
        n_val = n_total - n_dev - n_test  # Remainder goes to validation

        # Ensure at least 1 document per split if stratum is large enough
        if n_total >= 3:
            n_dev = max(1, n_dev)
            n_val = max(1, n_val)
            n_test = max(1, n_test)

            # Adjust if we allocated too many
            while n_dev + n_val + n_test > n_total:
                if n_val > 1:
                    n_val -= 1
                elif n_dev > 1:
                    n_dev -= 1
                else:
                    n_test -= 1

        # Assign documents
        idx = 0
        splits["development"].extend(pmcids[idx:idx + n_dev])
        idx += n_dev

        splits["validation"].extend(pmcids[idx:idx + n_val])
        idx += n_val

        splits["test"].extend(pmcids[idx:idx + n_test])

    return splits


def print_split_summary(
    splits: Dict[str, List[str]],
    ground_truth_dict: Dict[str, dict]
) -> None:
    """
    Print summary of the stratified splits.

    Args:
        splits: Dictionary with lists of PMCIDs for each split
        ground_truth_dict: Dictionary from load_all_ground_truth()
    """
    print("\n" + "=" * 70)
    print("STRATIFIED SPLIT SUMMARY (TADP-based)")
    print("=" * 70)

    total = sum(len(s) for s in splits.values())

    for split_name, pmcids in splits.items():
        print(f"\n{split_name.upper()} SET: {len(pmcids)} documents ({len(pmcids)/total*100:.1f}%)")

        # Count strata within this split
        strata_counts = Counter()
        category_counts = Counter()
        total_tadp = 0

        for pmcid in pmcids:
            doc = ground_truth_dict[pmcid]
            strata_counts[doc["stratum"]] += 1
            category_counts[doc["category"]] += 1
            total_tadp += doc["tadp"]

        mean_tadp = total_tadp / len(pmcids) if pmcids else 0

        print(f"  Mean TADP: {mean_tadp:.1f}")
        print(f"  Stratum distribution:")
        for stratum_name in STRATA_BINS.keys():
            count = strata_counts.get(stratum_name, 0)
            pct = (count / len(pmcids) * 100) if len(pmcids) > 0 else 0
            print(f"    {stratum_name}: {count} ({pct:.1f}%)")

        print(f"  Category distribution:")
        for category, count in sorted(category_counts.items(), key=lambda x: -x[1]):
            pct = (count / len(pmcids) * 100) if len(pmcids) > 0 else 0
            print(f"    {category}: {count} ({pct:.1f}%)")

## Output Functions

In [9]:
def save_split_assignments(
    splits: Dict[str, List[str]],
    ground_truth_dict: Dict[str, dict],
    output_folder: str,
    filename: str = "assay_validation_splits.json",
    pinned_documents: Dict[str, List[str]] = None
) -> str:
    """
    Save split assignments to JSON file for reproducibility.

    Args:
        splits: Dictionary with lists of PMCIDs for each split
        ground_truth_dict: Dictionary from load_all_ground_truth()
        output_folder: Path to output folder
        filename: Output filename
        pinned_documents: Dictionary of pinned documents

    Returns:
        Path to saved file
    """
    if pinned_documents is None:
        pinned_documents = PINNED_DOCUMENTS

    # Create output folder if needed
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    output_path = Path(output_folder) / filename

    # Build output data with metadata
    output_data = {
        "metadata": {
            "random_seed": RANDOM_SEED,
            "split_ratios": SPLIT_RATIOS,
            "strata_bins": {k: list(v) for k, v in STRATA_BINS.items()},
            "assay_fields": ASSAY_FIELDS,
            "total_documents": len(ground_truth_dict),
            "created_date": datetime.now().isoformat(),
            "stratification_metric": "TADP (Total Assay Data Points)"
        },
        "splits": {},
        "document_details": {}
    }

    # Add split data
    for split_name, pmcids in splits.items():
        output_data["splits"][split_name] = {
            "pmcids": pmcids,
            "count": len(pmcids),
            "pinned": [p for p in pmcids if p in pinned_documents.get(split_name, [])]
        }

    # Add document details
    for pmcid, doc_data in ground_truth_dict.items():
        # Find which split contains this document
        assigned_split = None
        for split_name, pmcids in splits.items():
            if pmcid in pmcids:
                assigned_split = split_name
                break

        output_data["document_details"][pmcid] = {
            "tadp": doc_data["tadp"],
            "stratum": doc_data["stratum"],
            "category": doc_data["category"],
            "isolates_with_linking_count": doc_data["isolates_with_linking_count"],
            "isolate_without_linking_count": doc_data["isolate_without_linking_count"],
            "clean_text_length": doc_data["clean_text_length"],
            "assigned_split": assigned_split
        }

    # Save to file
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, indent=2)

    print(f"\nSplit assignments saved to: {output_path}")

    return str(output_path)


def export_to_csv(
    ground_truth_dict: Dict[str, dict],
    splits: Dict[str, List[str]],
    output_folder: str,
    filename: str = "assay_ground_truth_details.csv"
) -> str:
    """
    Export detailed document information to CSV.

    Args:
        ground_truth_dict: Dictionary from load_all_ground_truth()
        splits: Dictionary with lists of PMCIDs for each split
        output_folder: Path to output folder
        filename: Output filename

    Returns:
        Path to saved file
    """
    import csv

    Path(output_folder).mkdir(parents=True, exist_ok=True)
    output_path = Path(output_folder) / filename

    # Create split lookup
    split_lookup = {}
    for split_name, pmcids in splits.items():
        for pmcid in pmcids:
            split_lookup[pmcid] = split_name

    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)

        # Write header
        writer.writerow([
            "PMCID",
            "TADP",
            "Stratum",
            "Category",
            "Split",
            "IWL_Count",
            "IWOL_Count",
            "Clean_Text_Length"
        ])

        # Write data rows (sorted by PMCID)
        for pmcid in sorted(ground_truth_dict.keys()):
            doc = ground_truth_dict[pmcid]
            writer.writerow([
                pmcid,
                doc["tadp"],
                doc["stratum"],
                doc["category"],
                split_lookup.get(pmcid, "unassigned"),
                doc["isolates_with_linking_count"],
                doc["isolate_without_linking_count"],
                doc["clean_text_length"]
            ])

    print(f"CSV export saved to: {output_path}")

    return str(output_path)

## Main Execution

In [10]:
def main():
    """
    Main execution function - analyse ground truth and create stratified splits.
    """
    print("=" * 70)
    print("ASSAY GROUND TRUTH ANALYSIS AND TADP-BASED STRATIFIED SAMPLING")
    print("=" * 70)

    # Load ground truth
    print(f"\nLoading ground truth from: {GROUND_TRUTH_FOLDER}")
    ground_truth_dict = load_all_ground_truth(GROUND_TRUTH_FOLDER)

    if not ground_truth_dict:
        print("ERROR: No ground truth files loaded. Check the folder path.")
        return None, None, None

    # Print example PMCIDs
    pmcid_list = list(ground_truth_dict.keys())
    print(f"\nLoaded {len(pmcid_list)} documents.")
    print("Example PMCIDs (first 5):")
    for pmcid in pmcid_list[:5]:
        doc = ground_truth_dict[pmcid]
        print(f"  {pmcid}: TADP={doc['tadp']}, Category={doc['category']}")

    # Analyse distribution
    statistics = analyse_tadp_distribution(ground_truth_dict)
    print_distribution_report(statistics)

    # Create stratified splits
    print("\n" + "=" * 70)
    print("CREATING STRATIFIED SPLITS")
    print("=" * 70)

    splits = create_stratified_three_set_split(ground_truth_dict)
    print_split_summary(splits, ground_truth_dict)

    # Save outputs
    save_split_assignments(splits, ground_truth_dict, OUTPUT_FOLDER)
    export_to_csv(ground_truth_dict, splits, OUTPUT_FOLDER)

    print("\n" + "=" * 70)
    print("ANALYSIS COMPLETE")
    print("=" * 70)

    return ground_truth_dict, statistics, splits

In [11]:
if __name__ == "__main__":
    ground_truth_dict, statistics, splits = main()

ASSAY GROUND TRUTH ANALYSIS AND TADP-BASED STRATIFIED SAMPLING

Loading ground truth from: /content/drive/MyDrive/AI6129/ground_truth
Found 227 JSON files in folder

Loaded 227 documents.
Example PMCIDs (first 5):
  PMC4672624: TADP=5, Category=NIOAI
  PMC4806280: TADP=2, Category=NIOAI
  PMC4824889: TADP=0, Category=IWOL
  PMC4859176: TADP=13, Category=IWL
  PMC4866840: TADP=4, Category=IWL

ASSAY GROUND TRUTH TADP DISTRIBUTION ANALYSIS

Total Documents: 227
Total Assay Data Points: 4737
Mean TADP per Document: 20.87
Min TADP: 0
Max TADP: 292
Documents with Zero TADP: 92

----------------------------------------------------------------------
TADP STRATIFICATION SUMMARY
----------------------------------------------------------------------
Stratum      TADP Range      Count      Percentage  
----------------------------------------------------------------------
zero         0               92         40.5%
low          1-10            50         22.0%
medium       11-50           54   

## Optional: Print PMCIDs by Stratum or Category

In [14]:
# Uncomment to print PMCIDs for specific strata
for stratum in statistics["strata_documents"]:
    print(f"\n{stratum.upper()} stratum ({len(statistics['strata_documents'][stratum])} documents):")
    for pmcid in statistics["strata_documents"][stratum][:10]:
        print(f"  {pmcid}")
    if len(statistics["strata_documents"][stratum]) > 10:
        print(f"  ... and {len(statistics['strata_documents'][stratum]) - 10} more")


ZERO stratum (92 documents):
  PMC4824889
  PMC5388333
  PMC6003729
  PMC6370545
  PMC7083327
  PMC7168671
  PMC7200987
  PMC7273606
  PMC7366320
  PMC7581672
  ... and 82 more

LOW stratum (50 documents):
  PMC4672624
  PMC4806280
  PMC4866840
  PMC4939201
  PMC5094244
  PMC5797408
  PMC5798080
  PMC5902092
  PMC5966692
  PMC6016173
  ... and 40 more

MEDIUM stratum (54 documents):
  PMC4859176
  PMC4944992
  PMC5832104
  PMC5938507
  PMC6307136
  PMC6329902
  PMC6744686
  PMC7008049
  PMC7497748
  PMC7603275
  ... and 44 more

HIGH stratum (31 documents):
  PMC4881965
  PMC6035434
  PMC8311177
  PMC8698551
  PMC8944838
  PMC8951033
  PMC9148033
  PMC9220226
  PMC9431532
  PMC9513250
  ... and 21 more


In [15]:
# Uncomment to print PMCIDs for specific categories
for category in statistics["category_documents"]:
    print(f"\n{category} category ({len(statistics['category_documents'][category])} documents):")
    for pmcid in statistics["category_documents"][category][:10]:
        print(f"  {pmcid}")
    if len(statistics["category_documents"][category]) > 10:
        print(f"  ... and {len(statistics['category_documents'][category]) - 10} more")


NIOAI category (21 documents):
  PMC4672624
  PMC4806280
  PMC5094244
  PMC5797408
  PMC5798080
  PMC5832104
  PMC5966692
  PMC6016173
  PMC6380132
  PMC6925778
  ... and 11 more

IWOL category (84 documents):
  PMC4824889
  PMC6003729
  PMC7083327
  PMC7168671
  PMC7200987
  PMC7273606
  PMC7366320
  PMC7581672
  PMC7705031
  PMC7712790
  ... and 74 more

IWL category (92 documents):
  PMC4859176
  PMC4866840
  PMC4881965
  PMC4939201
  PMC5388333
  PMC5902092
  PMC5938507
  PMC6035434
  PMC6329902
  PMC6370545
  ... and 82 more

IWL+IWOL category (30 documents):
  PMC4944992
  PMC6307136
  PMC8021795
  PMC9387262
  PMC9431532
  PMC9513250
  PMC6986767
  PMC6817352
  PMC5047355
  PMC8000398
  ... and 20 more
